# Lab 2: How hot is a typical July day, and does a bell curve describe it?

**Estimated time:** about 75 minutes

**Difficulty:** mathematics: Introductory to Developing; coding: Introductory (run and modify); science concepts: Developing; interpretation: Moderate

## What you will be able to do

1. Describe a set of daily temperatures with a center (mean) and a spread (standard deviation), and say in words what each number means physically.
2. Predict, before calculating, roughly where the mean falls, how wide the spread is, and whether the distribution leans to one side.
3. Test whether a Gaussian (bell curve) is a good description of real data, using a picture, the 68 and 95 percent rule, and skewness.
4. Explain why two cities can have similar averages but different distribution shapes, and propose a weather-based reason.

## What this lab assumes

- No calculus is needed for this lab. It uses averages and standard deviations only.
- You have seen a histogram. You do not need to have written code before; you will run and modify short cells.

## How this lab works

Each part follows the same pattern: **predict** what you expect, **calculate**
it, **visualize** it, then **explain** what you found. Write your prediction
before you run the code. Being wrong is useful information here; it is the
reason for the prediction step.

**Before you run the first cell:** Colab will show a popup warning that this
notebook "was not authored by Google." Colab shows this for every notebook
opened from GitHub, not just this one, and it does not mean anything is wrong.
Click "Run anyway" to continue.

## Step 0: Set up

In [ ]:
# Install the course library from GitHub. The @v0.3.0 pins this notebook to
# one tagged release, so everyone gets exactly the same code and data.
# Colab starts a fresh environment every session, so this line runs every
# time you open the notebook. That is normal, not an error.
# This line installs the course library. The address is not a web page, so there is no need to click it.

!pip install git+https://github.com/SaharMK22/qi_course.git@v0.3.0 --quiet

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import qi_course as qc

# Confirm which version of the library is running.
print("qi_course version:", qc.__version__)

## Step 1: The scientific question

You will work with the **daily high temperature** recorded at a real weather
station on every July day for more than a century. Every day in July has a
high, and no two Julys are the same.

> On a typical July day, how hot does it get? How much does that vary from day
> to day and from year to year? And can a single bell curve, described by just a
> mean and a standard deviation, summarize all of those days well?

Why July? A full year of daily highs is dominated by the seasonal cycle
(cold winters, hot summers), so it is not one bell curve. Restricting to a
single calendar month removes most of the seasonal swing, so the remaining
variation is mostly weather.

In [ ]:
# The stations that come with the course. Each is a real station with a
# daily record about a century long or more. "pct_tmax" is the percent of
# days that have a trustworthy daily high.
qc.list_stations()[["name", "state", "first_date", "last_date", "pct_tmax"]]

In [ ]:
# Choose a station and a month. STATION can be any piece of a name from the
# table above, for example "Portland" or "San Francisco".
STATION = "New Orleans"
MONTH = 7          # 7 means July. Later, try 1 (January).

## Step 2: Predict (do this before running any more code)

Write short answers in the cell below. Rough guesses are fine.

1. **Center:** What is the typical July daily high at this station, in degrees C?
   (Room temperature is about 20 C, a very hot day is about 40 C.)
2. **Spread:** On a typical July day, how far from that center might the high
   be? Within 1 degree, 3 degrees, 10 degrees?
3. **Shape:** Sketch the histogram you expect. Is it symmetric, or does it
   have a longer tail toward hot days or toward cool days?
4. **Comparison:** Would you expect this station's July average to be higher
   or lower than Portland, Oregon? Than San Francisco?

*Double-click this cell and type your predictions here:*

## Step 3: Look at the data before using it

Real records have imperfections. The next cell loads the station's complete
daily record and reports how complete it is.

In [ ]:
df = qc.load_station(STATION)
print(df.attrs["label"])

# One row per calendar day. NaN means "no trustworthy value that day".
# It is not zero. A day with a missing value is left out of every calculation.
print(df.head(3))

# How complete is the record?
coverage = qc.station_coverage(df)
print("\nRecord runs from", coverage["first_year"], "to", coverage["last_year"])
print("Percent of days present:", coverage["pct_present"])
print("Years with more than 10% of daily highs missing:", coverage["years_with_gaps"])

Two things to notice. First, some years have gaps. A station can be closed,
moved, or have its records lost, and the files simply have no value on those
days. Second, temperatures in this record were measured in whole degrees
Fahrenheit and then converted to Celsius, so the Celsius values come in steps
of about 0.6 degrees. That matters for the histogram below: bins of an
arbitrary width can catch two possible readings in one bar and one in the
next, which makes the bars uneven because of the rounding, not the weather.
The function `qc.temperature_bins` builds bins that each cover exactly two
degrees Fahrenheit to avoid this.

## Step 4: Calculate

Collect every daily high in your chosen month, across all years, then
summarize them.

In [ ]:
# All daily highs for the chosen month, in every year on record.
# Days with a missing value are dropped.
highs = qc.month_values(df, "tmax_c", MONTH)

stats = qc.descriptive_stats(highs)
print(f"Number of days:        {stats['n']}")
print(f"Mean (center):         {stats['mean']:.1f} C")
print(f"Std. dev. (spread):    {stats['std']:.2f} C")
print(f"Coolest / hottest day: {stats['min']:.1f} C / {stats['max']:.1f} C")
print(f"Skewness:              {stats['skewness']:+.2f}")

Compare these numbers with your predictions. If your guess for the mean was
off by more than a few degrees, what did you assume about this station's climate?

In [ ]:
# Check the standard deviation by hand on five days, so the number is not a mystery.
sample = highs.iloc[:5].values
print("Five daily highs:", sample)

mean_by_hand = sample.sum() / len(sample)
deviations = sample - mean_by_hand            # how far each day is from the mean
squared = deviations ** 2                     # squaring makes every term positive
std_by_hand = np.sqrt(squared.sum() / (len(sample) - 1))   # divide by n-1 for a sample

print(f"Mean by hand:               {mean_by_hand:.2f}")
print(f"Standard deviation by hand: {std_by_hand:.2f}")
print(f"numpy's answer:             {np.std(sample, ddof=1):.2f}   (should match)")

In [ ]:
# The 68-95 rule. For a bell curve, about 68% of values lie within 1 standard
# deviation of the mean and about 95% lie within 2. Does this station follow it?
for k, expected in [(1, "68%"), (2, "95%")]:
    print(f"Within {k} std of the mean: {qc.fraction_within(highs, k):.1%}   (bell curve: about {expected})")

# A bell curve is symmetric, so it puts about 2.3% of days beyond +2 std and
# about 2.3% beyond -2 std. Check each tail separately.
mean, std = stats["mean"], stats["std"]
print(f"\nHotter than mean + 2 std: {np.mean(highs > mean + 2*std):.1%}   (bell curve: about 2.3%)")
print(f"Cooler than mean - 2 std: {np.mean(highs < mean - 2*std):.1%}   (bell curve: about 2.3%)")

## Step 5: Visualize

Overlay the bell curve that has the same mean and standard deviation as your
data. If the data were exactly Gaussian, the bars would follow the curve.

In [ ]:
# Fit the bell curve: this only needs the mean and standard deviation.
grid, pdf, params = qc.fit_normal_pdf(highs)

# Bin edges that line up with whole degrees Fahrenheit (each bin is 2 F wide),
# so the rounding in the thermometer readings does not make the bars uneven.
bins = qc.temperature_bins(highs)

ax = qc.histogram_with_fit(
    highs, grid, pdf,
    title=f"{df.attrs['label']}: daily highs in month {MONTH}",
    xlabel="Daily high temperature (C)", bins=bins)

# Mark the center and the two-standard-deviation limits.
ax.axvline(params["mean"], color="black", linestyle=":")
ax.axvline(params["mean"] - 2 * params["std"], color="gray", linestyle="--")
ax.axvline(params["mean"] + 2 * params["std"], color="gray", linestyle="--")
plt.show()

## Step 6: Compare stations

One station tells you about one place. The next cell repeats the same
calculations for every bundled station so you can compare shapes, not just
averages.

In [ ]:
rows = []
fig, axes = plt.subplots(2, 2, figsize=(11, 7), sharex=False)
for ax, (name, label) in zip(axes.ravel(), zip(qc.list_stations()["name"], qc.list_stations()["label"])):
    d = qc.load_station(name)
    x = qc.month_values(d, "tmax_c", MONTH)
    s = qc.descriptive_stats(x)
    g, p, _ = qc.fit_normal_pdf(x)
    b = qc.temperature_bins(x)
    qc.histogram_with_fit(x, g, p, title=f"{label}  (skewness {s['skewness']:+.2f})",
                          xlabel="Daily high (C)", bins=b, ax=ax)
    rows.append({"station": label, "mean_C": s["mean"], "std_C": s["std"],
                 "skewness": s["skewness"]})
plt.tight_layout()
plt.show()

pd.DataFrame(rows).round(2)

## Step 7: Explain

Answer in a few sentences each. Use numbers and the pictures above.

1. **Center and spread.** Which station has the highest July average? Which has
   the largest spread? Give a physical reason (ocean nearby, humidity, clouds, elevation)
   for at least one of those differences.
2. **How Gaussian is it?** Rank the stations from "closest to a bell curve" to
   "farthest." What did you use to decide: the picture, the 68-95 percentages,
   the skewness, or the two tails? Did the methods agree?
3. **A subtle point.** A station can pass the 68-95 test and still be clearly
   lopsided. Find one station where that happens and describe what the tail
   check or histogram shows that the percentages hide.
4. **Physical reason for the shape.** Skewness above zero means a longer tail
   toward hot days; below zero means a longer tail toward cool days. For the
   most skewed station, suggest one weather process that could produce that
   tail. (You are proposing an explanation, not proving it. What data would
   you need to test it?)
5. **Your prediction.** Look back at your predictions in Step 2. Which one was
   furthest off, and what did you learn from the gap?

## Optional extension A: Is the climate changing, or is that just noise?

The record is long enough to ask whether July highs in the early years differ
from July highs in recent years. Daily values in the same month are not
independent (a hot spell lasts several days), so we compare **yearly means**:
one number per year.

In [ ]:
# One number per year: the average July high that year. Keep years with at
# least 20 days of data so a year with only a few days does not mislead.
by_year = highs.groupby(highs.index.year)
yearly_mean = by_year.mean()[by_year.count() >= 20]

middle = yearly_mean.index[len(yearly_mean) // 2]
early = yearly_mean[yearly_mean.index < middle]
late = yearly_mean[yearly_mean.index >= middle]

print(f"Early years ({early.index.min()}-{early.index.max()}): average {early.mean():.2f} C")
print(f"Later years ({late.index.min()}-{late.index.max()}): average {late.mean():.2f} C")
print(f"Difference: {late.mean() - early.mean():+.2f} C")
print(f"Typical year-to-year scatter (std of yearly means): {yearly_mean.std():.2f} C")

plt.figure(figsize=(8, 3.8))
plt.plot(yearly_mean.index, yearly_mean.values, "o-", markersize=3, linewidth=0.8)
plt.axhline(early.mean(), color="gray", linestyle="--", label="early average")
plt.axhline(late.mean(), color="black", linestyle="--", label="later average")
plt.xlabel("Year"); plt.ylabel("Average July high (C)"); plt.legend(); plt.grid(alpha=0.3)
plt.show()

Is the difference between the two halves large or small compared with the
year-to-year scatter? What other changes at a station, besides the climate,
could shift its recorded temperatures over a century (think about where the
thermometer is, what is around it, and how it is read)?

## Optional extension B: Use your own city

The bundled stations are a menu, not a limit. You can search NOAA's full
station list for long-record stations near any place in the United States.

1. Look up your city's latitude and longitude on any map website. In the cell below,
   west longitudes are negative.
2. Remove the `#` at the start of each line below and run the cell. The first
   search downloads NOAA's station list and can take a minute or two.
3. Choose a station id from the table whose record is at least 100 years long and
   close to your city. Copy its id (for example `USW00023160`).
4. Set `STATION` in Step 0 to that id and run the notebook again from Step 3.
   Read the coverage report before you trust the result.

In [ ]:
# nearby = qc.find_stations(lat=32.22, lon=-110.97, n=8, min_years=100)   # Tucson, as an example
# nearby

Remember that a weather station is not a city. Check the station's name and its
distance from where you live, and read the coverage report from Step 3.

## Optional extension C: Check an AI answer

Ask an AI assistant: "Why are San Francisco's July daily highs skewed toward hot days?"
Then test its explanation instead of accepting it. Does it make a prediction you
can check in this data (for example, a different skewness in January, or in a
station farther from the ocean)? Run that check and write two sentences: what the
AI claimed, what the data showed, and whether the explanation survived.

## If something breaks

The install cell is the one most likely to act up. If a later cell reports
that a function does not exist, use Runtime > Restart session, then run the
cells again from the top. For any other error, do not guess. Copy the exact
error text, note which cell it came from, and send both to your instructor or TA.

---

## Notes for the instructor (draft; delete this cell before sharing with students)

**Status:** first working draft using real NOAA GHCN-Daily records, for discussion. Time estimates are my own, not tested with students.

**Data:** four bundled stations pass a 100-year record with at least 90 percent of days present over 1926-2025. The set is geographically narrow (three West Coast stations and New Orleans); interior and desert stations are missing from this first pull. Adding a station only requires a new row in `stations.csv` and a new daily file.

**What students should find (July daily highs, all years):**

| Station | Mean (C) | Std (C) | Skewness | Within 1 std | Within 2 std | Beyond -2 std / +2 std |
|---|---|---|---|---|---|---|
| Los Angeles (downtown) | 28.1 | 3.00 | +0.55 | 69% | 96.0% | 0.6% / 3.4% |
| New Orleans (Audubon) | 33.2 | 2.02 | -0.56 | 70% | 95.6% | 3.8% / 0.6% |
| Portland, OR (Bull Run headworks) | 25.7 | 4.85 | +0.23 | 68% | 95.4% | 1.4% / 3.2% |
| San Francisco (downtown) | 18.8 | 3.08 | +1.46 | 75% | 94.7% | 0.0% / 5.3% |

For a bell curve the last three columns would read about 68%, 95%, and 2.3% / 2.3%.

**Intended discovery:** the 68-95 percentages are near Gaussian at most stations (San Francisco's 75 percent within 1 std is the exception), yet the tails are lopsided at every one: for San Francisco (downtown), skewness is +1.46, with 0.0% of days below -2 std but 5.3% above +2 std. The histograms show the skew directly. Summary numbers can pass while the shape fails.

**Likely misconceptions:** (1) NaN treated as zero, which the library prevents; (2) reading a standard deviation as a range that contains all values; (3) assuming a symmetric picture means normal; (4) comparing stations without noticing different record lengths or gaps.

**Plausible physical explanations to discuss (hypotheses, not tested here):** San Francisco's marine layer caps most July highs while rare offshore-flow days run far hotter; New Orleans' occasional storm and cloud days pull highs down, giving a cool-side tail.

**Real-data caveats:** daily maximum is recorded in whole degrees Fahrenheit (0.6 C steps), hence the bins that are a whole number of degrees F wide. Some stations are cooperative observer sites with time-of-observation changes. Station moves and urban growth can create apparent trends; extension A is the place to raise this.

**Open questions for Dale:**
1. Should every student use the same station first, then compare, or choose freely from the start?
2. Is skewness the right level of detail here, or should it wait for a later extremes lab?
3. Do we want the optional AI-critique extension in the first student version?
4. Which stations should the bundled menu include? Tucson is the obvious one but is not in this first pull (I have not yet diagnosed why); a shorter record with a clear label may be better than omitting it.